# MLP - potencial solar e eolico com gold

Este notebook usa exclusivamente `data/gold/inmet_pe_daily.csv` para treinar uma MLP multi-saida que estima potencial esperado diario em kWh para uma estacao teorica fixa.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import mlflow.sklearn
import pandas as pd
from threadpoolctl import threadpool_limits

from src.modeling.gold_energy import (
    FEATURE_COLUMNS,
    TARGET_COLUMNS,
    assert_no_forbidden_features,
    describe_search_space,
    evaluate_predictions,
    fit_final_model,
    load_gold_daily,
    make_temporal_cv_splits,
    mlp_base_estimator,
    mlp_refinement_space,
    mlp_search_space,
    predict_future_for_station,
    prepare_energy_modeling_table,
    split_feature_target_metadata,
    temporal_train_test_split,
    to_jsonable,
    train_random_search,
)
from src.modeling.training_config import (
    BLAS_THREADS,
    CPU_WORKERS,
    ENERGY_CONFIG,
    FUTURE_DATE,
    FUTURE_STATION_CODE,
    MLFLOW_EXPERIMENT_NAME,
    TEST_YEAR_FRACTION,
)

print(f"Raiz do projeto: {PROJECT_ROOT}")

In [ ]:
# Configuracoes compartilhadas ficam em src/modeling/training_config.py.
# A semente aleatoria fica local no notebook, para permitir reprodutibilidade por modelo.
RANDOM_STATE = 42
N_ITER_RANDOM = 120
N_ITER_REFINEMENT = 40

assert CPU_WORKERS > 0, "CPU_WORKERS deve ser positivo."
assert BLAS_THREADS > 0, "BLAS_THREADS deve ser positivo."
print(f"CPU_WORKERS={CPU_WORKERS}; BLAS_THREADS={BLAS_THREADS}; threads planejadas={CPU_WORKERS * BLAS_THREADS}")


In [ ]:
daily_gold = load_gold_daily(PROJECT_ROOT)
modeling_table = prepare_energy_modeling_table(daily_gold, ENERGY_CONFIG)
X, y, metadata = split_feature_target_metadata(modeling_table)
assert_no_forbidden_features(list(X.columns))

(
    X_train,
    X_test,
    y_train,
    y_test,
    train_metadata,
    test_metadata,
    train_years,
    test_years,
) = temporal_train_test_split(X, y, metadata, test_year_fraction=TEST_YEAR_FRACTION)
cv_splits = make_temporal_cv_splits(train_metadata)

print(f"Linhas gold usadas: {len(modeling_table):,}")
print(f"Variaveis de entrada: {FEATURE_COLUMNS}")
print(f"Alvos: {TARGET_COLUMNS}")
print(f"Anos treino/validacao: {train_years}")
print(f"Anos teste final: {test_years}")
print(f"Folds temporais no treino: {len(cv_splits)}")


In [ ]:
mlp_estimator = mlp_base_estimator(random_state=RANDOM_STATE)
mlp_space = mlp_search_space()
print("Espaco amplo MLP:", describe_search_space(mlp_space))

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="mlp_gold_energy"):
    mlflow.log_param("model_family", "MLPRegressor")
    mlflow.log_param("cpu_workers", CPU_WORKERS)
    mlflow.log_param("blas_threads", BLAS_THREADS)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_param("n_iter_random", N_ITER_RANDOM)
    mlflow.log_param("n_iter_refinement", N_ITER_REFINEMENT)
    mlflow.log_param("train_years", ",".join(map(str, train_years)))
    mlflow.log_param("test_years", ",".join(map(str, test_years)))
    mlflow.log_param("features", ",".join(FEATURE_COLUMNS))
    mlflow.log_dict(to_jsonable(ENERGY_CONFIG), "energy_config.json")
    mlflow.log_dict(to_jsonable(mlp_space), "search_space_random.json")

    with threadpool_limits(limits=BLAS_THREADS):
        broad_search, broad_seconds = train_random_search(
            mlp_estimator,
            mlp_space,
            X_train,
            y_train,
            cv_splits,
            n_iter=N_ITER_RANDOM,
            n_jobs=CPU_WORKERS,
            random_state=RANDOM_STATE,
        )

    mlp_ref_space = mlp_refinement_space(broad_search.best_params_)
    mlflow.log_metric("broad_search_seconds", broad_seconds)
    mlflow.log_metric("broad_best_balanced_negative_nrmse", broad_search.best_score_)
    mlflow.log_dict(to_jsonable(broad_search.best_params_), "best_params_broad.json")
    mlflow.log_dict(to_jsonable(mlp_ref_space), "search_space_refinement.json")

    with threadpool_limits(limits=BLAS_THREADS):
        refine_search, refine_seconds = train_random_search(
            mlp_estimator,
            mlp_ref_space,
            X_train,
            y_train,
            cv_splits,
            n_iter=N_ITER_REFINEMENT,
            n_jobs=CPU_WORKERS,
            random_state=RANDOM_STATE + 1,
        )

    with threadpool_limits(limits=BLAS_THREADS):
        final_model, final_fit_seconds = fit_final_model(
            mlp_estimator,
            refine_search.best_params_,
            X_train,
            y_train,
        )

    test_predictions = final_model.predict(X_test)
    metrics_table, final_metrics = evaluate_predictions(y_test, test_predictions)
    for metric_name, metric_value in final_metrics.items():
        if pd.notna(metric_value):
            mlflow.log_metric(metric_name, float(metric_value))
    mlflow.log_metric("refine_search_seconds", refine_seconds)
    mlflow.log_metric("final_fit_seconds", final_fit_seconds)
    mlflow.log_metric("refine_best_balanced_negative_nrmse", refine_search.best_score_)
    mlflow.log_dict(to_jsonable(refine_search.best_params_), "best_params_refinement.json")
    mlflow.sklearn.log_model(final_model, artifact_path="model")

metrics_table

In [ ]:
test_results = test_metadata.copy()
for index, target in enumerate(TARGET_COLUMNS):
    test_results[f"{target}_actual"] = y_test[target].to_numpy()
    test_results[f"{target}_pred"] = test_predictions[:, index]
test_results["hybrid_generation_kwh_day_actual"] = test_results[[f"{target}_actual" for target in TARGET_COLUMNS]].sum(axis=1)
test_results["hybrid_generation_kwh_day_pred"] = test_results[[f"{target}_pred" for target in TARGET_COLUMNS]].sum(axis=1)

display(metrics_table)
display(test_results.head(20))


In [ ]:
# Preencha FUTURE_STATION_CODE e FUTURE_DATE em src/modeling/training_config.py.
if FUTURE_STATION_CODE is None or FUTURE_DATE is None:
    print("Preencha FUTURE_STATION_CODE e FUTURE_DATE para gerar inferencia futura.")
else:
    station_prediction, future_ranking = predict_future_for_station(
        final_model,
        modeling_table,
        station_code=FUTURE_STATION_CODE,
        future_date=FUTURE_DATE,
    )
    display(station_prediction)
    display(future_ranking.head(20))
